# Note:
- The data from the [Chicago Data Portal](https://data.cityofchicago.org/browse?category=Public+Safety&sortBy=most_accessed&page=1&pageSize=20) and Crime Data set were enriched using multiple datasets from the portal. We initially stored them in the PostgreSQL database to generate the enriched dataset by joining multiple datasets using the_geom, but we discovered inconsistencies in the police beat, district, and sector fields. All missing fields were determined using multiple fields to generate the most accurate information, but there may be errors during the data wrangling process.

- We designate the primary Crime dataset as the authoritative source of truth. To ensure consistency and address missing values, we perform internal imputation using data from other sources to fill corresponding NaN entries in location-based fields.

In [1]:
# import libraries
from platform import python_version
import sys
import time
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
import numpy as np
import re

# python source path
sys.path.append('../Src/')

# random seed
_RANDOM_STATE = 1776
    
# python
import utils
import geo
import geo_dict

# Create a dictionary of versions
versions = {
    "Python": sys.version.split()[0],
    "Pandas": pd.__version__,
    "NumPy": np.__version__,
    "Pyarrow": pa.__version__
}

# Display as a clean DataFrame
df_versions = pd.DataFrame(list(versions.items()), columns=['Library', 'Version'])
print(df_versions)

# Use a single Arrow string & int64 type instance to save memory
arrow_string = pd.ArrowDtype(pa.string())
arrow_float64 = pd.ArrowDtype(pa.float64())
arrow_int64 = pd.ArrowDtype(pa.int64())
arrow_int8 = pd.ArrowDtype(pa.int8())

# for low cardinality categorical columns (2–128 unique values)
arrow_cat8  = pd.ArrowDtype(pa.dictionary(
                  index_type=pa.int8(), 
                  value_type=pa.string()
              ))

# capture time
start = time.time()

   Library Version
0   Python  3.13.9
1   Pandas   2.3.3
2    NumPy   2.3.4
3  Pyarrow  22.0.0


## Read Data
- The Chicago Crime Data contains Crime, Arrest, IUCR, Neighborhood, and Police Beat datasets from the Chicago Crime Portal.

In [2]:
# display all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
# reset options
# pd.reset_option('display.max_columns')

In [3]:
# load using pyarrow for performance (crime data is joined between crime & neighborhood & police using geom)
df_crime = pd.read_csv("../Data/chicago_crimes_export.csv", engine="pyarrow", dtype_backend="pyarrow")
# Police Info
police = pd.read_csv("../Data/police_beats_export.csv", engine="pyarrow", dtype_backend="pyarrow")

In [4]:
df_crime.head()

,case_number,date,block,iucr,primary_description,secondary_description,index_code,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_code,year,updated_on,fbi_code,zip_code,zip_code_area,primary_neighborhood,secondary_neighborhood,neighborhood_area,p_district,p_sector,p_beat,ca_community_code,ca_community_name,ca_community_area,latitude,longitude,x_coordinate,y_coordinate
0,JJ214830,2025-04-03 13:00:00,100XX W BALMORAL AVE,0810,THEFT,OVER $500,I,THEFT,OVER $500,AIRPORT BUILDING NON-TERMINAL - NON-SECURE AREA,f,f,1654,16,41,76,2025,2026-03-14 15:41:39,06,60666,282085138.571,O'Hare,OHARE,371835607.687,16,5,1654,76,OHARE,371835607.687,41.976182,-87.876421,1108491,1934242
1,JJ240422,2025-04-03 13:00:00,021XX W 71ST ST,0486,BATTERY,DOMESTIC BATTERY SIMPLE,N,BATTERY,DOMESTIC BATTERY SIMPLE,RESIDENCE - GARAGE,f,t,735,7,17,67,2025,2026-03-14 15:41:39,08B,60636,104114706.716,Englewood,ENGLEWOOD,173600015.009,7,3,735,67,WEST ENGLEWOOD,87947691.9478,41.764765,-87.677703,1163114,1857553
2,JJ205003,2025-04-03 13:00:00,007XX N CENTRAL PARK AVE,0560,ASSAULT,SIMPLE,N,ASSAULT,SIMPLE,RESIDENCE,f,t,1112,11,27,23,2025,2026-03-14 15:41:39,08A,60624,99418122.6738,Humboldt Park,HUMBOLDT PARK,125010425.593,11,2,1121,23,HUMBOLDT PARK,100480876.502,41.894273,-87.716279,1152251,1904667
3,JJ205029,2025-04-03 13:00:00,006XX N WELLS ST,0870,THEFT,POCKET-PICKING,I,THEFT,POCKET-PICKING,STREET,f,f,1832,18,42,8,2025,2026-03-14 15:41:39,06,60654,15869961.5669,River North,RIVER NORTH,38766442.5194,18,3,1832,8,NEAR NORTH SIDE,76675895.9728,41.893645,-87.634123,1174621,1904610
4,JJ204673,2025-04-03 13:00:00,119XX S HALSTED ST,0460,BATTERY,SIMPLE,N,BATTERY,SIMPLE,CTA BUS,f,f,524,5,21,53,2025,2026-03-14 15:41:39,08B,60643,207706232.893,West Pullman,WEST PULLMAN,99365198.0822,5,2,524,53,WEST PULLMAN,99365198.0822,41.67736,-87.641894,1173139,1825780


## Describe Data

In [5]:
df_crime.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
case_number,8543704,8543089,HZ140230,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date,8543704,NaN,NaN,NaN,2011-09-18 00:52:29,2001-01-01 00:00:00,2005-06-27 19:24:45,2010-07-10 02:00:00,2017-07-28 07:00:00,2026-04-25 00:00:00,NaN
block,8543704,65797,001XX N STATE ST,17201,NaN,NaN,NaN,NaN,NaN,NaN,NaN
iucr,8543704,418,0820,683850,NaN,NaN,NaN,NaN,NaN,NaN,NaN
primary_description,8524333,32,THEFT,1804998,NaN,NaN,NaN,NaN,NaN,NaN,NaN
secondary_description,8524333,369,SIMPLE,1005413,NaN,NaN,NaN,NaN,NaN,NaN,NaN
index_code,8524333,2,N,5046029,NaN,NaN,NaN,NaN,NaN,NaN,NaN
primary_type,8543704,34,THEFT,1814892,NaN,NaN,NaN,NaN,NaN,NaN,NaN
description,8543704,569,SIMPLE,1005413,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location_description,8527736,218,STREET,2232912,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data Wrangle
- Chicago's `IUCR` codes (Illinois Uniform Crime Reporting) are four-digit codes for classifying crimes, with the Chicago Police Department (CPD) using over 400, including FBI Index Offenses (homicide, robbery, theft) and Non-Index offenses (vandalism, weapons violations)
- Chicago has `50 wards`, each represented by an alderperson, with boundaries redrawn every eight years
- The Chicago Police Department (CPD) divides the city into `22 Districts`, which are further broken down into smaller patrol zones called `Beats`, with specific 4-digit numbers for each area
- Chicago is divided into `77 official Community Areas.

In [6]:
df_crime.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8543704 entries, 0 to 8543703
Data columns (total 34 columns):
 #   Column                  Non-Null Count    Dtype                
---  ------                  --------------    -----                
 0   case_number             8543704 non-null  string[pyarrow]      
 1   date                    8543704 non-null  timestamp[s][pyarrow]
 2   block                   8543704 non-null  string[pyarrow]      
 3   iucr                    8543704 non-null  string[pyarrow]      
 4   primary_description     8524333 non-null  string[pyarrow]      
 5   secondary_description   8524333 non-null  string[pyarrow]      
 6   index_code              8524333 non-null  string[pyarrow]      
 7   primary_type            8543704 non-null  string[pyarrow]      
 8   description             8543704 non-null  string[pyarrow]      
 9   location_description    8527736 non-null  string[pyarrow]      
 10  arrest                  8543704 non-null  string[pyarr

In [7]:
# mask operation
mask = df_crime['community_code'] != df_crime['ca_community_code']
# count the difference
mask.sum()
# display
df_crime.loc[mask, ['community_code','ca_community_code']].head()

,community_code,ca_community_code
20,34,60
65,20,23
147,13,14
161,6,3
173,73,72


In [8]:
# mask operation
mask = (
    df_crime['primary_neighborhood'].fillna('').str.upper()
    != df_crime['secondary_neighborhood'].fillna('')
)
# count the difference
print(mask.sum())
# display
df_crime.loc[mask, ['primary_neighborhood','secondary_neighborhood']].sample(n=10)

3162049


,primary_neighborhood,secondary_neighborhood
8145043,South Shore,"SOUTH SHORE, GRAND CROSSING"
2386618,South Deering,SOUTHEAST SIDE
6304706,Douglas,BRONZEVILLE
8257532,North Park,"NORTH PARK,ALBANY PARK"
4874017,Chatham,"CHATHAM,BURNSIDE"
4979990,Montclare,"MONTCLARE, GALEWOOD"
4229,Chatham,"CHATHAM,BURNSIDE"
772701,Chicago Lawn,"MARQUETTE PARK,GAGE PARK"
2529309,Archer Heights,"ARCHER HEIGHTS,WEST ELSDON"
3257394,Brighton Park,"BRIGHTON PARK,MCKINLEY PARK"


### Remove Duplicates

In [9]:
# Remove Duplicate Case Numbers
cleaned_dict = utils.deduplicate_and_report(df_crime)
# Sanity Check
pd.DataFrame(cleaned_dict['df_cleaned']['case_number'].describe()).T

DEDUPLICATION SUMMARY (Arrow Backend)
Original Rows : 8,543,704
Cleaned Rows  : 8,543,089
Total Deleted : 615

TOP 10 DELETIONS BY CASE NUMBER:
case_number  records_deleted
   HJ590004                5
   HZ140230                5
   JC470284                4
   JE266473                4
   HS256531                4
   HP296582                4
   JJ309322                3
   HJ756295                3
   HJ104730                3
   HY346207                3


,count,unique,top,freq
case_number,8543089,8543089,01G050460,1


In [10]:
# Picks 5 values from the cleaned_dict report DataFrame
random_5 = np.random.choice(cleaned_dict['report']['case_number'], size=5, replace=False)
# display
df_crime.loc[df_crime.case_number.isin(random_5),]

,case_number,date,block,iucr,primary_description,secondary_description,index_code,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_code,year,updated_on,fbi_code,zip_code,zip_code_area,primary_neighborhood,secondary_neighborhood,neighborhood_area,p_district,p_sector,p_beat,ca_community_code,ca_community_name,ca_community_area,latitude,longitude,x_coordinate,y_coordinate
911559,JE309374,2021-07-23 17:50:00,033XX W DOUGLAS BLVD,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,ALLEY,t,f,1021,10,24,29,2021,2022-09-01 15:42:17,01A,60623,155285530.844,North Lawndale,NORTH LAWNDALE,89487422.0244,10,2,1021,29,NORTH LAWNDALE,89487422.0242,41.862917,-87.709616,1154148,1893254
915198,JE309374,2021-07-21 18:31:00,033XX W DOUGLAS BLVD,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,ALLEY,t,f,1021,10,24,29,2021,2022-09-19 15:41:05,01A,60623,155285530.844,North Lawndale,NORTH LAWNDALE,89487422.0244,10,2,1021,29,NORTH LAWNDALE,89487422.0242,41.862917,-87.709616,1154148,1893254
1401258,JC279072,2019-05-26 06:48:00,013XX W HASTINGS ST,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,CHA PARKING LOT,t,f,1233,12,25,28,2019,2022-09-19 15:41:05,01A,60608,176505462.842,"Little Italy, UIC","LITTLE ITALY, UIC",71376244.1225,12,3,1233,28,NEAR WEST SIDE,158492466.554,41.864278,-87.65966,1167752,1893853
1401262,JC279072,2019-05-26 06:35:00,013XX W HASTINGS ST,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,CHA GROUNDS,t,f,1233,12,25,28,2019,2022-09-19 15:41:05,01A,60608,176505462.842,"Little Italy, UIC","LITTLE ITALY, UIC",71376244.1225,12,3,1233,28,NEAR WEST SIDE,158492466.554,41.864278,-87.65966,1167752,1893853
2398003,HY402004,2015-08-29 09:42:00,052XX S LOREL AVE,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,HOUSE,f,f,814,8,23,56,2015,2022-09-19 15:41:05,01A,60638,166166338.996,Garfield Ridge,MIDWAY AIRPORT,117890778.429,8,1,814,56,GARFIELD RIDGE,117890778.429,41.797828,-87.75634,1141580,1869447
2398004,HY402004,2015-08-29 09:42:00,052XX S LOREL AVE,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,HOUSE,f,f,814,8,23,56,2015,2022-09-19 15:41:05,01A,60638,166166338.996,Garfield Ridge,MIDWAY AIRPORT,117890778.429,8,1,814,56,GARFIELD RIDGE,117890778.429,41.797828,-87.75634,1141580,1869447
4666753,HP669416,2008-11-06 17:13:00,015XX W 63RD ST,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,ALLEY,f,f,725,7,16,67,2008,2022-09-19 15:41:05,01A,60636,104114706.716,Englewood,ENGLEWOOD,173600015.009,7,1,713,67,WEST ENGLEWOOD,87947691.9478,41.779503,-87.66218,1167307,1862956
4666754,HP669416,2008-11-06 17:13:00,015XX W 63RD ST,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,ALLEY,f,f,725,7,16,67,2008,2022-09-01 15:42:17,01A,60636,104114706.716,Englewood,ENGLEWOOD,173600015.009,7,1,713,67,WEST ENGLEWOOD,87947691.9478,41.779503,-87.66218,1167307,1862956
4998185,HP168112,2008-02-09 21:20:00,027XX S DRAKE AVE,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,STREET,t,t,1032,10,22,30,2008,2022-09-01 15:42:17,01A,60623,155285530.844,Little Village,LITTLE VILLAGE,127998297.819,10,3,1032,30,SOUTH LAWNDALE,127998297.867,41.841944,-87.713505,1153144,1885604
4998186,HP168112,2008-02-09 21:20:00,027XX S DRAKE AVE,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,STREET,t,t,1032,10,22,30,2008,2022-09-19 15:41:05,01A,60623,155285530.844,Little Village,LITTLE VILLAGE,127998297.819,10,3,1032,30,SOUTH LAWNDALE,127998297.867,41.841944,-87.713505,1153144,1885604


### Deep Cleaned Copy

In [11]:
# Deep Copy 
df_crime = cleaned_dict['df_cleaned'].copy()

### The Timeline Definition
* To ensure the analysis is accurate, we define the three eras based on global lockdown patterns:
    * Pre-COVID: January 2001 – February 2020
    * COVID Era: March 2020 – December 31, 2022
    * Post-COVID: January 2023 – Present

In [12]:
# Define boundary timestamps
covid_start      = pd.Timestamp('2020-03-01')
post_covid_start = pd.Timestamp('2023-01-01')

# Define conditions (evaluated top to bottom)
conditions = [
    df_crime['date'] < covid_start,          # Pre-COVID
    df_crime['date'] < post_covid_start      # COVID
]

# Corresponding labels
choices = ['pre_covid', 'covid']

# Apply vectorized selection
df_crime['era'] = (
    np.select(conditions, choices, default='post_covid')
    )

# Cast to memory-efficient categorical dtype
# Only 3 unique values -> int8 index is sufficient
df_crime['era'] = df_crime['era'].astype(arrow_cat8)

# Sanity Check
print(df_crime['era'].value_counts())
print(f"\nEra null count: {df_crime['era'].isna().sum()}")
print(f"\nDate range per era:")
print(df_crime.groupby('era')['date'].agg(['min', 'max']))

era
pre_covid     7092863
post_covid     826157
covid          624069
Name: count, dtype: int64[pyarrow]

Era null count: 0

Date range per era:
                            min                  max
era                                                 
covid       2020-03-01 00:00:00  2022-12-31 23:55:00
post_covid  2023-01-01 00:00:00  2026-04-25 00:00:00
pre_covid   2001-01-01 00:00:00  2020-02-29 23:59:00


### Invalid (x_coordinate, y_coordinate, latitude, longitude)

In [13]:
# Condition 1: Invalid State Plane coordinates
mask_xy = (df_crime.x_coordinate == 0) | (df_crime.y_coordinate == 0)

# Standard Chicago Bounding Box (WGS84)
LAT_MIN, LAT_MAX = 41.60, 42.05
LON_MIN, LON_MAX = -87.94, -87.50

# Condition 2: Invalid WGS84 coordinates (outside Chicago bounding box)
mask_latlon = (
    (df_crime.latitude  < LAT_MIN) |
    (df_crime.latitude  > LAT_MAX) |
    (df_crime.longitude < LON_MIN) |
    (df_crime.longitude > LON_MAX)
)

# Diagnostic
print(f"Invalid x/y:      {mask_xy.sum():,}")
print(f"Invalid lat/lon:  {mask_latlon.sum():,}")
print(f"Overlap:          {(mask_xy & mask_latlon).sum():,}")
print(f"Total unique:     {(mask_xy | mask_latlon).sum():,}")

# Null out invalid coordinates (float cols → np.nan)
df_crime.loc[mask_xy,     ['x_coordinate', 'y_coordinate']] = np.nan
df_crime.loc[mask_latlon, ['latitude', 'longitude']]        = np.nan

Invalid x/y:      149
Invalid lat/lon:  149
Overlap:          149
Total unique:     149


### Invalid community_code

In [14]:
# Invalid community_code
mask = (df_crime.community_code == 0)
mask.sum()

76

In [15]:
# Update community_code
df_crime.loc[mask, ['community_code']] = pd.NA

### Check for Duplicates

In [16]:
# get dupes
dupes = df_crime.duplicated(keep='last')
# any duplicates
if dupes.any():
    print(f"Number of Duplicates: {dupes.sum():,}")
else:
    print("No Duplicates")

No Duplicates


### Unique Values

In [17]:
# display number of unique values
for i in df_crime.columns:
    print(f"{i}: {df_crime[i].nunique():,}")

case_number: 8,543,089
date: 3,584,258
block: 65,796
iucr: 418
primary_description: 32
secondary_description: 369
index_code: 2
primary_type: 34
description: 569
location_description: 218
arrest: 2
domestic: 2
beat: 305
district: 24
ward: 50
community_code: 77
year: 26
updated_on: 7,658
fbi_code: 26
zip_code: 59
zip_code_area: 59
primary_neighborhood: 98
secondary_neighborhood: 78
neighborhood_area: 98
p_district: 23
p_sector: 5
p_beat: 275
ca_community_code: 77
ca_community_name: 77
ca_community_area: 77
latitude: 911,676
longitude: 911,080
x_coordinate: 79,380
y_coordinate: 130,455
era: 3


### Update: location_description

In [18]:
# Replaces : , -, and multi-spaces with a single space
def clean_locations(series):
    out = (
        series.str.replace(r'[\s:,-]+', ' ', regex=True)  # Combined delimiters to space
              .str.replace(r'\s*/\s*', '/', regex=True)   # Fix slashes
              .str.strip()
    )
    
    return out
# Apply regex
df_crime['location_description'] = clean_locations(df_crime['location_description'])

# Dictionary mapping (Vectorized replace)
mapping = {
    'NURSING HOME/RETIREMENT HOME': 'NURSING/RETIREMENT HOME', 
    'OTHER RAILROAD PROP/TRAIN DEPOT': 'OTHER RAILROAD PROPERTY/TRAIN DEPOT',
    'PARKING LOT/GARAGE(NON.RESID.)': 'PARKING LOT/GARAGE (NON RESIDENTIAL)',
    'POLICE FACILITY/VEH PARKING LOT': 'POLICE FACILITY/VEHICLE PARKING LOT',
    'POOLROOM': 'POOL ROOM', 
    'RESIDENCE YARD (FRONT/BACK)': 'RESIDENTIAL YARD (FRONT/BACK)',
    'TAXICAB': 'TAXI CAB',
    'VEHICLE OTHER RIDE SERVICE': 'VEHICLE OTHER RIDE SHARE SERVICE (LYFT UBER ETC.)',
    'VEHICLE OTHER RIDE SHARE SERVICE (E.G. UBER LYFT)': 'VEHICLE OTHER RIDE SHARE SERVICE (LYFT UBER ETC.)'
}

# Apply mapping first
df_crime['location_description'] = df_crime['location_description'].replace(mapping)

### New Feature(s)

In [19]:
# Add Month & Day of the Week
months = ['January', 'February', 'March', 'April', 'May', 'June', 
          'July', 'August', 'September', 'October', 'November', 'December']
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Extract integers and map them
# .dt.month returns 1-12, so we subtract 1 for 0-based indexing
df_crime['month'] = np.array(months)[df_crime['date'].dt.month.values - 1]

# .dt.dayofweek returns 0-6 (0 is Monday)
df_crime['day_of_week'] = np.array(days)[df_crime['date'].dt.dayofweek.values]

# convert to pyarrow
df_crime['month'] = df_crime['month'].astype(arrow_string)
df_crime['day_of_week'] = df_crime['day_of_week'].astype(arrow_string)

In [20]:
# Map to strings first
df_crime['quarter'] = df_crime['date'].dt.quarter.map({1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4'}).astype(arrow_string)
# combine
df_crime['year_quarter'] = (df_crime['year'].astype("string[pyarrow]") + "-" + df_crime['quarter'])
# Force pyarrow datatype
df_crime['year_quarter'] = df_crime['year_quarter'].astype(arrow_string)

| Interval (Inclusive, Exclusive) | Mathematical Notation | Label        | Hours Included    |
|--------------------------------|----------------------|--------------|-------------------|
| 1st: 0 to 4                    | \([0, 4)\)          | Late Night   | 0, 1, 2, 3       |
| 2nd: 4 to 8                    | \([4, 8)\)          | Early Morning| 4, 5, 6, 7       |
| 3rd: 8 to 12                   | \([8, 12)\)         | Morning      | 8, 9, 10, 11     |
| 4th: 12 to 16                  | \([12, 16)\)        | Afternoon    | 12, 13, 14, 15   |
| 5th: 16 to 20                  | \([16, 20)\)        | Evening      | 16, 17, 18, 19   |
| 6th: 20 to 24                  | \([20, 24)\)        | Night        | 20, 21, 22, 23   |

In [21]:
# Get the hours as a PyArrow-backed integer
hours = df_crime['date'].dt.hour.values

# Use np.digitize for ultra-fast binning (vectorized)
# bins: [0, 4, 8, 12, 16, 20, 24]
# digitize returns 1 for 0-3, 2 for 4-7, etc.
bin_indices = np.digitize(hours, bins=[4, 8, 12, 16, 20])

# Map indices to labels
time_labels = np.array(['Late Night', 'Early Morning', 'Morning', 'Afternoon', 'Evening', 'Night'])
df_crime['time_of_day'] = time_labels[bin_indices]

# Final cast to string[pyarrow]
df_crime['time_of_day'] = df_crime['time_of_day'].astype(arrow_string)

### FBI Code Mapping

In [22]:
# display fbi_code
print(sorted(df_crime['fbi_code'].unique()))

['01A', '01B', '02', '03', '04A', '04B', '05', '06', '07', '08A', '08B', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '22', '24', '26']


In [23]:
# determine specific values in the data that are missing from the mapping dictionary
df_crime.loc[~df_crime["fbi_code"].isin(geo_dict.fbi_codes.keys()), "fbi_code" ].unique()

<ArrowExtensionArray>
[]
Length: 0, dtype: string[pyarrow]

In [24]:
df_crime[["fbi_code_desc", "fbi_index_code"]] = pd.DataFrame({
    "fbi_code_desc": df_crime["fbi_code"].map(lambda x: geo_dict.fbi_codes[x]["desc"]),
    "fbi_index_code": df_crime["fbi_code"].map(lambda x: geo_dict.fbi_codes[x]["is_index"])
})

# convert to arrow datatype
df_crime["fbi_code_desc"] = df_crime["fbi_code_desc"].astype(arrow_string)
df_crime["fbi_index_code"] = df_crime["fbi_index_code"]

In [25]:
# df_crime[['iucr','primary_description','secondary_description','description','fbi_code']][df_crime['fbi_code'] == '01A'].sample(5)
df_crime[['fbi_code_desc','description', 'location_description', 'domestic', 'fbi_index_code']].sample(n=5, random_state=_RANDOM_STATE)

,fbi_code_desc,description,location_description,domestic,fbi_index_code
6005870,Miscellaneous Non-Index Offenses,ANIMAL ABUSE/NEGLECT,OTHER,f,False
2904737,Vandalism,TO PROPERTY,RESIDENTIAL YARD (FRONT/BACK),f,False
5835722,Simple Battery,SIMPLE,STREET,f,False
3503885,Larceny – Theft,$500 AND UNDER,CHA APARTMENT,f,True
4490856,Drug Abuse Violations,POSS: CANNABIS 30GMS OR LESS,ALLEY,f,False


### Datatype Change (Boolean)

In [26]:
# check for unique values
df_crime[['arrest','domestic', 'fbi_index_code']].apply(lambda s: s.unique())

,arrest,domestic,fbi_index_code
0,t,f,False
1,f,t,True


In [27]:
# convert to pyarrow boolean
df_crime[['arrest','domestic']] = (
    df_crime[['arrest','domestic']] # domestic: Domestic violence
        .apply(lambda col: col.map({'t': True, 'f': False}))
        .astype(bool)
)

### Feature Information:
* A Chicago `ward` is one of 50 legislative districts, each represented by an elected Alderman on the City Council, serving as local government branches to provide city services, manage development, and reflect community demographics, with boundaries redrawn every 10 years based on census data.
* The `district` feature refers to the city's 22 police districts, which are geographic areas used to organize crime data.
* The `beat` feature in Chicago crime data identifies the smallest geographic police area (a beat) where a crime occurred.
* The `sector` refers to a specific geographic division used by the Chicago Police Department (CPD), where several smaller `beats` (police patrol areas) are grouped together to form a sector, which then rolls up into a larger `district`, providing a layered geographic context for analyzing crime trends.
* The `Community Area` feature refers to one of 77 distinct, officially defined, and geographically stable neighborhoods used for urban planning and statistical analysis. This feature allows categorizing crime incidents by location, enabling trend analysis and identifying high-crime areas.

In [28]:
# Remove columns (Assign the result back to the main variable)
remove_cols = ["p_district", "p_beat"]
df_crime = df_crime.drop(columns=remove_cols)

# Rename 'p_sector' to 'sector'
df_crime = df_crime.rename(columns={'p_sector': 'sector'})

# Verify the change
print(df_crime.columns)

Index(['case_number', 'date', 'block', 'iucr', 'primary_description',
       'secondary_description', 'index_code', 'primary_type', 'description',
       'location_description', 'arrest', 'domestic', 'beat', 'district',
       'ward', 'community_code', 'year', 'updated_on', 'fbi_code', 'zip_code',
       'zip_code_area', 'primary_neighborhood', 'secondary_neighborhood',
       'neighborhood_area', 'sector', 'ca_community_code', 'ca_community_name',
       'ca_community_area', 'latitude', 'longitude', 'x_coordinate',
       'y_coordinate', 'era', 'month', 'day_of_week', 'quarter',
       'year_quarter', 'time_of_day', 'fbi_code_desc', 'fbi_index_code'],
      dtype='object')


### Update Datatypes & Fill
- Add padding if required

In [29]:
# display
police.head()

,district,sector,beat
0,1,1,111
1,1,1,112
2,1,1,113
3,1,1,114
4,1,2,121


In [30]:
# change to string and must be three char length
cols = police.columns.to_list()

# iterate cols
for col in cols:

    if col in ['district']:
        # Fill NAs and convert to a standard string for the zfill operation
        # https://www.chicagopolice.org/statistics-data/crime-statistics/
        police[col] = police[col].astype("string").str.zfill(3)
    elif col in ['beat']:
         police[col] = police[col].astype("string").str.zfill(4)
    else:
        # Fill NAs and convert to a standard string
        police[col] = police[col].astype("string")
        
    # Force ArrowDtype
    police[col] = police[col].astype(arrow_string)

police[cols].sample(5)

,district,sector,beat
49,004,3,0434
11,002,1,0211
211,018,2,1823
0,001,1,0111
6,001,2,0123


In [31]:
# change to string and must be three char length
cols = ['district', 'beat', 'ward', 'sector', 'community_code', 'year']

# iterate cols
for col in cols:

    if col in ['district']:
        # Fill NAs and convert to a standard string for the zfill operation
        # https://www.chicagopolice.org/statistics-data/crime-statistics/
        df_crime[col] = df_crime[col].astype("string").str.zfill(3)
    elif col in ['ward', 'community_code']:
        df_crime[col] = df_crime[col].astype("string").str.zfill(2)
    elif col in ['beat']:
         df_crime[col] = df_crime[col].astype("string").str.zfill(4)
    else:
        # Fill NAs and convert to a standard string
        df_crime[col] = df_crime[col].astype("string")
        
    # Force ArrowDtype
    df_crime[col] = df_crime[col].astype(arrow_string)

df_crime[cols].sample(5)

,district,beat,ward,sector,community_code,year
2405918,018,1833,42,3,08,2015
6456683,016,1614,41,<NA>,76,2004
7683598,009,0934,20,3,61,2002
5124082,025,2535,30,3,23,2007
963867,025,2511,36,1,19,2021


### Update Invalid district / New column (district_loc)

In [32]:
# display district
utils.wrap_unique(df_crime, 'district')

[001, 002, 003, 004, 005, 006, 007, 008, 009, 010, 011, 012, 014, 015, 016, 017,
018, 019, 020, 021, 022, 024, 025, 031]
::::: Unique Count: 24 (+ 47 nulls)


In [33]:
# Invalid district (21, 31)
mask = df_crime.district.isin(['021', '031'])
mask.sum()

np.int64(281)

In [34]:
# Update Invalid district
df_crime.loc[mask, 'district'] = pd.NA 

In [35]:
# display district
utils.wrap_unique(df_crime, 'district')

[001, 002, 003, 004, 005, 006, 007, 008, 009, 010, 011, 012, 014, 015, 016, 017,
018, 019, 020, 022, 024, 025]
::::: Unique Count: 22 (+ 328 nulls)


In [36]:
utils.wrap_unique(police, 'district')

[001, 002, 003, 004, 005, 006, 007, 008, 009, 010, 011, 012, 014, 015, 016, 017,
018, 019, 020, 022, 024, 025]
::::: Unique Count: 22


In [37]:
utils.wrap_unique(police, 'sector')

[1, 2, 3, 5]
::::: Unique Count: 4


In [38]:
# display sector
utils.wrap_unique(df_crime, 'sector')

[0, 1, 2, 3, 5]
::::: Unique Count: 5 (+ 117,393 nulls)


According to a search, the Chicago Police Department (CPD) currently operates 277 active, specialized beats across 22 districts, employing a community-policing model in which 8–9 officers are assigned to patrol specific areas for at least a year. Our Police table in the Chicago Data Hub lists 274 beats, but our crime data contains 305, and we assume that some beats were consolidated due to Chuicago Police Department overhaul over the past 20 years.

In [39]:
print("Number of Beats in the Crime Data:" , df_crime.beat.nunique())
print("Number of Beats in the Police Data:" , police.beat.nunique())

Number of Beats in the Crime Data: 305
Number of Beats in the Police Data: 274


In [40]:
# Compare
crime_set = set(df_crime.beat.to_list())
police_set = set(police.beat.to_list())

# check sets
print("Does crime_set contain all of police_set: " ,crime_set.issuperset(police_set))
print("Does police_set contain all of crime_set: " ,police_set.issuperset(crime_set))

Does crime_set contain all of police_set:  True
Does police_set contain all of crime_set:  False


In [41]:
# Set Compare (symmetric difference)
print(sorted(crime_set ^ police_set))
print("Mis-match Beat Count:", len(crime_set ^ police_set))

['0134', '0310', '0430', '1311', '1312', '1313', '1322', '1323', '1324', '1331', '1332', '1333', '1650', '2111', '2112', '2113', '2122', '2123', '2124', '2131', '2132', '2133', '2311', '2312', '2313', '2322', '2323', '2324', '2331', '2332', '2333']
Mis-match Beat Count: 31


In [42]:
print(sorted(crime_set - police_set))
print("Extra Beat Count from crime_set:", len(crime_set ^ police_set))

['0134', '0310', '0430', '1311', '1312', '1313', '1322', '1323', '1324', '1331', '1332', '1333', '1650', '2111', '2112', '2113', '2122', '2123', '2124', '2131', '2132', '2133', '2311', '2312', '2313', '2322', '2323', '2324', '2331', '2332', '2333']
Extra Beat Count from crime_set: 31


In [43]:
# Initialize
invalid_beat = list(crime_set - police_set)

In [44]:
df_crime.head()

,case_number,date,block,iucr,primary_description,secondary_description,index_code,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_code,year,updated_on,fbi_code,zip_code,zip_code_area,primary_neighborhood,secondary_neighborhood,neighborhood_area,sector,ca_community_code,ca_community_name,ca_community_area,latitude,longitude,x_coordinate,y_coordinate,era,month,day_of_week,quarter,year_quarter,time_of_day,fbi_code_desc,fbi_index_code
8265318,01G050460,2001-01-24 20:45:00,072XX S RIDGELAND AV,1811,NARCOTICS,POSSESS - CANNABIS 30 GRAMS OR LESS,N,NARCOTICS,POSS: CANNABIS 30GMS OR LESS,SIDEWALK,True,False,0324,003,<NA>,<NA>,2001,2015-08-17 15:03:40,18,60649,80526075.8505,South Shore,"SOUTH SHORE, GRAND CROSSING",81812716.3904,2,43,SOUTH SHORE,81812716.3958,41.764219,-87.582549,1189075,1857566,pre_covid,January,Wednesday,Q1,2001-Q1,Night,Drug Abuse Violations,False
7080787,03J493690,2003-07-12 17:00:00,075XX S DOBSON AVE,0890,THEFT,FROM BUILDING,I,THEFT,FROM BUILDING,APARTMENT,False,False,0624,006,08,69,2003,2015-08-17 15:03:40,06,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,pre_covid,July,Saturday,Q3,2003-Q3,Evening,Larceny – Theft,True
6396203,04X245238,2004-12-13 21:15:00,006XX N RIDGEWAY AVE,2024,NARCOTICS,POSSESS - HEROIN (WHITE),N,NARCOTICS,POSS: HEROIN(WHITE),SIDEWALK,True,False,1122,011,27,23,2004,2018-02-28 15:56:25,18,60624,99418122.6738,Humboldt Park,HUMBOLDT PARK,125010425.593,2,23,HUMBOLDT PARK,100480876.502,41.892451,-87.719888,1151273,1903996,pre_covid,December,Monday,Q4,2004-Q4,Night,Drug Abuse Violations,False
5818733,07C115980,2006-03-31 09:15:00,026XX N NARRAGANSETT AVE,0610,BURGLARY,FORCIBLE ENTRY,I,BURGLARY,FORCIBLE ENTRY,APARTMENT,False,False,2512,025,29,19,2006,2018-02-28 15:56:25,05,60707,48519709.6539,Belmont Cragin,"BELMONT CRAGIN,HERMOSA",109099407.211,1,19,BELMONT CRAGIN,109099414.689,41.928096,-87.78561,1133296,1916864,pre_covid,March,Friday,Q1,2006-Q1,Morning,Burglary,True
5312174,07HN36467,2007-05-25 14:51:00,022XX N LA CROSSE AVE,1812,NARCOTICS,POSSESS - CANNABIS MORE THAN 30 GRAMS,N,NARCOTICS,POSS: CANNABIS MORE THAN 30GMS,RESIDENCE,True,False,2522,025,31,19,2007,2018-02-28 15:56:25,18,60639,127476051.26,Belmont Cragin,"BELMONT CRAGIN,HERMOSA",109099407.211,2,19,BELMONT CRAGIN,109099414.689,41.921066,-87.747452,1143697,1914371,pre_covid,May,Friday,Q2,2007-Q2,Afternoon,Drug Abuse Violations,False


In [45]:
# columns to display
cols = ['year', 'primary_neighborhood',	'secondary_neighborhood', 'neighborhood_area', 'community_code', 
        'ca_community_code','ca_community_name', 'ca_community_area', 'ward', 'district', 'sector', 'beat',
        'zip_code', 'x_coordinate', 'y_coordinate', 'latitude', 'longitude']
mask = df_crime.beat.isin(invalid_beat)
print(f"Possible Invalid 'beat' from Cime Data Count:  {(mask.sum()):,}")
bad_beat = df_crime.loc[mask, cols].copy()
bad_beat.head()

Possible Invalid 'beat' from Cime Data Count:  351,327


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
8272314,2001,Douglas,BRONZEVILLE,46004621.137,<NA>,35,DOUGLAS,46004621.1581,<NA>,002,1,2112,60616,1178109,1882924,41.834059,-87.621973
7918746,2001,Oakland,"OAKLAND,KENWOOD",16913961.0408,<NA>,36,OAKLAND,16913961.0408,<NA>,002,1,2123,60653,1183840,1878113,41.820725,-87.601095
8010810,2001,Kenwood,"KENWOOD,OAKLAND",29071741.9283,39,39,KENWOOD,29071741.9283,04,002,2,2124,60615,1182421,1872504,41.805367,-87.606475
7906901,2001,Near South Side,NEAR SOUTH SIDE,34252582.7003,<NA>,33,NEAR SOUTH SIDE,49769639.4541,<NA>,002,3,2111,60616,1177107,1890696,41.855409,-87.625414
8294051,2001,Uptown,UPTOWN,65095642.836,<NA>,3,UPTOWN,65095642.7289,<NA>,019,1,2311,60640,1167921,1930894,41.965917,-87.657969


In [46]:
# count rows by year
bad_beat['year'].value_counts().sort_index()

year
2001    36982
2002    36251
2003    35330
2004    34972
2005    34714
2006    32333
2007    30002
2008    29480
2009    25284
2010    24096
2011    22549
2012     9201
2020        1
2023       91
2024       41
Name: count, dtype: int64[pyarrow]

In [47]:
mask = bad_beat.year.isin(['2024', '2023', '2020'])
bad_beat.loc[mask, cols].drop_duplicates().head()

,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
528421,2023,O'Hare,OHARE,371835607.687,76,76,OHARE,371835607.687,41,016,5,1650,60666,1108491,1934242,41.976182,-87.876421
1239683,2020,<NA>,<NA>,<NA>,76,<NA>,<NA>,<NA>,41,016,<NA>,1650,<NA>,<NA>,<NA>,<NA>,<NA>
250619,2023,<NA>,<NA>,<NA>,76,<NA>,<NA>,<NA>,41,016,<NA>,1650,<NA>,<NA>,<NA>,<NA>,<NA>
307120,2024,O'Hare,OHARE,371835607.687,76,76,OHARE,371835607.687,41,016,5,1650,60666,1108491,1934242,41.976182,-87.876421
298874,2024,O'Hare,OHARE,371835607.687,76,76,OHARE,371835607.687,41,016,5,1650,60666,1108424,1934249,41.976202,-87.876667


##### According to [Chicago District Map](chrome-extension://efaidnbmnnnibpcajpcglclefindmkaj/https://chicagocop.com/wp-content/uploads/Chicago-Police-Department-Citywide-Area-District-Beat-Map-2009-March.pdf), beat 1650 does not exist; it's possible it was merged into beat 1651, since that is the O'Hare community. We will not change the beat column and will assume it is accurate, since we are unable to verify from the source whether any changes or consolidations occurred with the Chicago Police Department overall.

### **Note:**
* Chicago’s crime data is recorded across a complex framework of overlapping jurisdictions, ranging from political districts to social neighborhoods. At the administrative level, the Chicago Police Department operates through a hierarchy of Districts and Beats. A Beat is the smallest geographic unit, assigned to a specific patrol car for community policing, while multiple Beats are grouped into a District managed by a central precinct. For example, Beats 2511, 2514, and 2521 all fall under the jurisdiction of District 025. Because these boundaries are drawn based on population density and response times rather than cultural history, they rarely align perfectly with the city’s social fabric.
* To provide a more stable lens for analysis, researchers utilize the city’s 77 Community Areas. Established in the 1920s by the University of Chicago, these fixed boundaries remain unchanged by political redistricting or postal updates, allowing for consistent longitudinal tracking of crime trends over decades. In contrast, Chicago’s 50 Wards are political entities redrawn every ten years to ensure equal population representation.
* Ultimately, "neighborhood" designations like "Albany Park" or "Irving Park" reflect social and historical identities rather than law-enforcement jurisdictions. Because these residential areas are often too large for a single patrol car to cover, a single neighborhood is frequently split across multiple Police Beats. This misalignment means that a single criminal incident may be categorized differently depending on whether the analyst is looking through a political (Ward), statistical (Community Area), or operational (Police District) lens.

### **NaN Note:**
- We designate the primary Crime dataset as the authoritative source of truth. To ensure consistency and address missing values, we perform internal imputation by using data from other sources to fill the corresponding NaN entries in the location-based fields.

#### Update sector

In [48]:
# Invert the mapping
cpd_sector = {
    dist: sector
    for sector, dists in geo_dict.cpd_sector.items() # Outer loop: looping over sector: list_of_districts
    for dist in dists # Inner loop: looping over each district inside that list
}

# Intentional overwrite: sector is derived deterministically from district
# using the cpd_sector dictionary (source:https://www.chicagopolice.org/statistics-data/crime-statistics/)
# sector is represented as Area
# update sector (note: it's not correct for all the years)
df_crime['sector'] = (
    df_crime['district']
        .map(cpd_sector)
).astype(arrow_cat8)

In [49]:
# New column
df_crime['district_location'] = df_crime.district.map(geo_dict.cpd_districts)
df_crime['district_location'] = df_crime['district_location'].astype(arrow_cat8)

### NaNs

In [50]:
# display
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,543,089) ---
                         Count Percentage
ward                    614800    7.1965%
community_code          613787    7.1846%
secondary_neighborhood  118627    1.3886%
ca_community_name       118627    1.3886%
primary_neighborhood    118627    1.3886%
neighborhood_area       118627    1.3886%
ca_community_area       118627    1.3886%
ca_community_code       118627    1.3886%
zip_code                118568    1.3879%
zip_code_area           118568    1.3879%
y_coordinate             96633    1.1311%
x_coordinate             96633    1.1311%
longitude                96633    1.1311%
latitude                 96633    1.1311%
primary_description      19371    0.2267%
secondary_description    19371    0.2267%
index_code               19371    0.2267%
location_description     15968    0.1869%
sector                     328    0.0038%
district                   328    0.0038%
district_location          328    0.0038%


### Inital Impute

In [51]:
def get_sample_report(data_df: pd.DataFrame, bool_mask: pd.Series, cols: list=cols, msg = True, n_sample: int = 5, seed: int = _RANDOM_STATE):
    """
    Displays a sample of the dataframe based on a mask.
    If there are fewer than 5 available rows, it displays all. Otherwise, displays n_sample.
    """
    # 1. Filter the data based on the mask
    filtered_df = data_df.loc[bool_mask, cols]
    available_count = len(filtered_df)

    # 2. Display counts from the mask
    print(f"::::: Search: {(bool_mask.sum()):,} :::::\n")
    
    
    # 3. Logic: If count < 5, take all. Else, take n_sample (default 5)
    if available_count < 5:
        sample_df = filtered_df
    else:
        sample_df = filtered_df.sample(n=min(n_sample, available_count), random_state=seed)

    # 4. Capture index and display
    sample_indices = sample_df.index.tolist()

    if msg:
        print(f"--- Showing {len(sample_df)} of {available_count:,} eligible rows ---")

    return sample_indices

#### longitude & latitude

In [52]:
# mask
mask = (
        df_crime.latitude.isna() & 
        df_crime.longitude.isna() &
        df_crime.beat.notna() 
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Search: 96,633 :::::

--- Showing 5 of 96,633 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
5945800,2005,<NA>,<NA>,<NA>,54,<NA>,<NA>,<NA>,09,005,2,0533,<NA>,<NA>,<NA>,<NA>,<NA>
7497075,2002,<NA>,<NA>,<NA>,01,<NA>,<NA>,<NA>,49,024,3,2424,<NA>,<NA>,<NA>,<NA>,<NA>
3898010,2010,<NA>,<NA>,<NA>,06,<NA>,<NA>,<NA>,46,019,3,1925,<NA>,<NA>,<NA>,<NA>,<NA>
7476658,2002,<NA>,<NA>,<NA>,08,<NA>,<NA>,<NA>,43,018,3,1822,<NA>,<NA>,<NA>,<NA>,<NA>
1786379,2017,<NA>,<NA>,<NA>,19,<NA>,<NA>,<NA>,30,025,5,2514,<NA>,<NA>,<NA>,<NA>,<NA>


In [53]:
# Composite key
keys = ['beat']
update_cols = ['latitude', 'longitude']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 96,633
Rows updated:       96,633
Rows not updated:   0
Values imputed:     193,266


In [54]:
df_crime.loc[idx, cols]

,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
5945800,2005,<NA>,<NA>,<NA>,54,<NA>,<NA>,<NA>,09,005,2,0533,<NA>,<NA>,<NA>,41.654917,-87.604963
7497075,2002,<NA>,<NA>,<NA>,01,<NA>,<NA>,<NA>,49,024,3,2424,<NA>,<NA>,<NA>,42.019337,-87.680511
3898010,2010,<NA>,<NA>,<NA>,06,<NA>,<NA>,<NA>,46,019,3,1925,<NA>,<NA>,<NA>,41.947048,-87.646931
7476658,2002,<NA>,<NA>,<NA>,08,<NA>,<NA>,<NA>,43,018,3,1822,<NA>,<NA>,<NA>,41.900769,-87.643107
1786379,2017,<NA>,<NA>,<NA>,19,<NA>,<NA>,<NA>,30,025,5,2514,<NA>,<NA>,<NA>,41.930482,-87.757325


#### x_coordinate & y_coordinate

In [55]:
# mask
mask = (
        df_crime.latitude.notna() & 
        df_crime.longitude.notna() &
        df_crime.x_coordinate.isna() &
        df_crime.y_coordinate.isna() 
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Search: 96,633 :::::

--- Showing 5 of 96,633 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
5945800,2005,<NA>,<NA>,<NA>,54,<NA>,<NA>,<NA>,09,005,2,0533,<NA>,<NA>,<NA>,41.654917,-87.604963
7497075,2002,<NA>,<NA>,<NA>,01,<NA>,<NA>,<NA>,49,024,3,2424,<NA>,<NA>,<NA>,42.019337,-87.680511
3898010,2010,<NA>,<NA>,<NA>,06,<NA>,<NA>,<NA>,46,019,3,1925,<NA>,<NA>,<NA>,41.947048,-87.646931
7476658,2002,<NA>,<NA>,<NA>,08,<NA>,<NA>,<NA>,43,018,3,1822,<NA>,<NA>,<NA>,41.900769,-87.643107
1786379,2017,<NA>,<NA>,<NA>,19,<NA>,<NA>,<NA>,30,025,5,2514,<NA>,<NA>,<NA>,41.930482,-87.757325


In [56]:
# Composite key
keys = ['latitude', 'longitude']
update_cols = ['x_coordinate', 'y_coordinate']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 96,633
Rows updated:       96,633
Rows not updated:   0
Values imputed:     193,266


In [57]:
# Display
df_crime.loc[idx, cols]

,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
5945800,2005,<NA>,<NA>,<NA>,54,<NA>,<NA>,<NA>,09,005,2,0533,<NA>,1183297,1817685,41.654917,-87.604963
7497075,2002,<NA>,<NA>,<NA>,01,<NA>,<NA>,<NA>,49,024,3,2424,<NA>,1161642,1950313,42.019337,-87.680511
3898010,2010,<NA>,<NA>,<NA>,06,<NA>,<NA>,<NA>,46,019,3,1925,<NA>,1170978,1924042,41.947048,-87.646931
7476658,2002,<NA>,<NA>,<NA>,08,<NA>,<NA>,<NA>,43,018,3,1822,<NA>,1172154,1907186,41.900769,-87.643107
1786379,2017,<NA>,<NA>,<NA>,19,<NA>,<NA>,<NA>,30,025,5,2514,<NA>,1140987,1917784,41.930482,-87.757325


#### ward & primary_neighborhood & secondary_neighborhood & neighborhood_area & community_code & ca_community_code & ca_community_name & ca_community_area

In [58]:
# mask
mask = (
    df_crime.year.notna() &
    df_crime.x_coordinate.notna() & 
    df_crime.y_coordinate.notna() &
    df_crime.latitude.notna() &
    df_crime.longitude.notna() &
        df_crime.ward.isna()
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Search: 614,800 :::::

--- Showing 5 of 614,800 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
8185975,2001,North Lawndale,NORTH LAWNDALE,89487422.0244,<NA>,29,NORTH LAWNDALE,89487422.0242,<NA>,010,4,1014,60623,1152615,1891091,41.857011,-87.715301
7893378,2001,Lincoln Park,LINCOLN PARK,67401451.3717,<NA>,7,LINCOLN PARK,88316400.4728,<NA>,018,3,1813,60614,1170418,1910905,41.911012,-87.649375
8215756,2001,Lower West Side,LOWER WEST SIDE,81550723.8231,<NA>,31,LOWER WEST SIDE,81550723.6377,<NA>,012,3,1222,60608,1166665,1889615,41.852672,-87.663772
7849094,2001,Woodlawn,WOODLAWN,40515739.083,<NA>,42,WOODLAWN,57815179.512,<NA>,003,1,0313,60637,1180766,1864640,41.783825,-87.612786
8291979,2001,West Ridge,WEST RIDGE,98429094.8621,<NA>,2,WEST RIDGE,98429094.8621,<NA>,024,3,2411,60645,1153890,1945254,42.005614,-87.709174


In [59]:
# Composite key
keys = ['year', 'x_coordinate', 'y_coordinate', 'latitude', 'longitude', 
        'community_code', 'district', 'sector', 'beat', 'zip_code']
update_cols = ['ward', 'primary_neighborhood', 'secondary_neighborhood', 'neighborhood_area', 
               'community_code', 'ca_community_code', 'ca_community_name', 'ca_community_area']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 614,800
Rows updated:       614,744
Rows not updated:   56
Values imputed:     1,285,710


In [60]:
# Verify
df_crime.loc[idx, cols]

,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
8185975,2001,North Lawndale,NORTH LAWNDALE,89487422.0244,02,29,NORTH LAWNDALE,89487422.0242,50,010,4,1014,60623,1152615,1891091,41.857011,-87.715301
7893378,2001,Lincoln Park,LINCOLN PARK,67401451.3717,02,7,LINCOLN PARK,88316400.4728,50,018,3,1813,60614,1170418,1910905,41.911012,-87.649375
8215756,2001,Lower West Side,LOWER WEST SIDE,81550723.8231,02,31,LOWER WEST SIDE,81550723.6377,50,012,3,1222,60608,1166665,1889615,41.852672,-87.663772
7849094,2001,Woodlawn,WOODLAWN,40515739.083,02,42,WOODLAWN,57815179.512,50,003,1,0313,60637,1180766,1864640,41.783825,-87.612786
8291979,2001,West Ridge,WEST RIDGE,98429094.8621,02,2,WEST RIDGE,98429094.8621,50,024,3,2411,60645,1153890,1945254,42.005614,-87.709174


#### community_code & primary_neighborhood & secondary_neighborhood & neighborhood_area & ca_community_code & ca_community_name & ca_community_area & zip_code & zip_code_area

In [61]:
# mask
mask = (
    df_crime.year.notna() &
    df_crime.x_coordinate.notna() & 
    df_crime.y_coordinate.notna() &
    df_crime.latitude.notna() &
    df_crime.longitude.notna() &
        df_crime.community_code.isna()
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Search: 1,597 :::::

--- Showing 5 of 1,597 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
4853081,2008,"Little Italy, UIC","LITTLE ITALY, UIC",71376244.1225,<NA>,28,NEAR WEST SIDE,158492466.554,02,012,3,1233,60608,1168892,1894548,41.866161,-87.655455
3699365,2011,Garfield Park,GARFIELD PARK,89976069.5947,<NA>,27,EAST GARFIELD PARK,53883220.8462,28,011,4,1124,60624,1154551,1899349,41.879634,-87.707974
5134204,2007,Hermosa,"BELMONT CRAIGIN,HERMOSA",32602059.4055,<NA>,20,HERMOSA,32602059.4055,31,025,5,2522,60641,1147419,1919614,41.935383,-87.733642
4240931,2009,"Little Italy, UIC","LITTLE ITALY, UIC",71376244.1225,<NA>,28,NEAR WEST SIDE,158492466.554,02,012,3,1222,60612,1163868,1896632,41.871987,-87.67384
5405164,2007,Garfield Park,GARFIELD PARK,89976069.5947,<NA>,27,EAST GARFIELD PARK,53883220.8462,28,011,4,1124,60624,1154551,1899349,41.879634,-87.707974


In [62]:
# Composite key
keys = ['year', 'x_coordinate', 'y_coordinate', 'latitude', 'longitude', 
        'district', 'sector', 'beat', 'zip_code']
update_cols = ['community_code', 'primary_neighborhood', 'secondary_neighborhood', 'neighborhood_area',
               'ca_community_code', 'ca_community_name', 'ca_community_area', 'zip_code', 'zip_code_area']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 1,597
Rows updated:       285
Rows not updated:   1,312
Values imputed:     1,629


#### community_code & primary_neighborhood & secondary_neighborhood & neighborhood_area & ca_community_code & ca_community_name & ca_community_area & zip_code & zip_code_area

In [63]:
# mask
mask = (
    df_crime.year.notna() &
    df_crime.x_coordinate.notna() & 
    df_crime.y_coordinate.notna() &
    df_crime.latitude.notna() &
    df_crime.longitude.notna() &
    (
        df_crime.primary_neighborhood.isna() |
        df_crime.neighborhood_area.isna() |
        df_crime.secondary_neighborhood.isna()
    )
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Search: 108,663 :::::

--- Showing 5 of 108,663 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
4674337,2008,<NA>,<NA>,<NA>,06,<NA>,<NA>,<NA>,46,019,3,2324,<NA>,1168396,1924119,41.947316,-87.656419
4524959,2009,<NA>,<NA>,<NA>,07,<NA>,<NA>,<NA>,43,018,3,1812,<NA>,1170558,1914520,41.920929,-87.648754
815546,2022,<NA>,<NA>,<NA>,22,<NA>,<NA>,<NA>,32,014,5,1414,<NA>,1156620,1915710,41.924488,-87.699933
4150771,2010,<NA>,<NA>,<NA>,17,<NA>,<NA>,<NA>,36,016,5,1631,<NA>,1125811,1925761,41.952638,-87.812916
1805079,2017,<NA>,<NA>,<NA>,66,<NA>,<NA>,<NA>,15,008,1,0831,<NA>,1159278,1857086,41.763563,-87.691776


In [64]:
# Composite key
keys = ['year', 'x_coordinate', 'y_coordinate', 'latitude', 'longitude',
        'district', 'sector', 'beat']
update_cols = ['community_code', 'primary_neighborhood', 'secondary_neighborhood', 'neighborhood_area',
               'ca_community_code', 'ca_community_name', 'ca_community_area', 'zip_code', 'zip_code_area']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 108,663
Rows updated:       9,945
Rows not updated:   98,718
Values imputed:     79,560


#### ca_community_code & ca_community_name & ca_community_area

In [65]:
# mask
mask = (
    df_crime.year.notna() &
    df_crime.x_coordinate.notna() & 
    df_crime.y_coordinate.notna() &
    df_crime.latitude.notna() &
    df_crime.longitude.notna() &
        (
            df_crime.ca_community_code.isna() |
             df_crime.ca_community_name.isna() |
             df_crime.ca_community_area.isna()
        )
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Search: 98,718 :::::

--- Showing 5 of 98,718 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
4927413,2008,<NA>,<NA>,<NA>,25,<NA>,<NA>,<NA>,36,025,5,2513,<NA>,1127853,1909983,41.909307,-87.805767
4588237,2009,<NA>,<NA>,<NA>,31,<NA>,<NA>,<NA>,25,012,3,1234,<NA>,1162459,1889720,41.853049,-87.679206
8354409,2026,<NA>,<NA>,<NA>,01,<NA>,<NA>,<NA>,49,024,3,2422,<NA>,1163126,1950347,42.019399,-87.675049
7044739,2003,<NA>,<NA>,<NA>,26,<NA>,<NA>,<NA>,28,011,4,1133,<NA>,1152765,1896840,41.872784,-87.714598
7515972,2002,<NA>,<NA>,<NA>,30,<NA>,<NA>,<NA>,22,010,4,1013,<NA>,1152077,1886651,41.844838,-87.717393


In [66]:
# Composite key
keys = ['year', 'x_coordinate', 'y_coordinate', 'latitude', 'longitude', 
        'zip_code', 'district', 'sector', 'beat']
update_cols = ['community_code', 'primary_neighborhood', 'secondary_neighborhood', 'neighborhood_area',
               'ca_community_code', 'ca_community_name', 'ca_community_area']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 98,718
Rows updated:       98,647
Rows not updated:   71
Values imputed:     591,882


In [67]:
# Verify
df_crime.loc[idx, cols]

,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
4927413,2008,West Ridge,WEST RIDGE,98429094.8621,25,2,WEST RIDGE,98429094.8621,36,025,5,2513,<NA>,1127853,1909983,41.909307,-87.805767
4588237,2009,West Ridge,WEST RIDGE,98429094.8621,31,2,WEST RIDGE,98429094.8621,25,012,3,1234,<NA>,1162459,1889720,41.853049,-87.679206
8354409,2026,West Ridge,WEST RIDGE,98429094.8621,01,2,WEST RIDGE,98429094.8621,49,024,3,2422,<NA>,1163126,1950347,42.019399,-87.675049
7044739,2003,West Ridge,WEST RIDGE,98429094.8621,26,2,WEST RIDGE,98429094.8621,28,011,4,1133,<NA>,1152765,1896840,41.872784,-87.714598
7515972,2002,West Ridge,WEST RIDGE,98429094.8621,30,2,WEST RIDGE,98429094.8621,22,010,4,1013,<NA>,1152077,1886651,41.844838,-87.717393


#### community_code & primary_neighborhood & secondary_neighborhood & neighborhood_area & ca_community_code & ca_community_name & ca_community_area & zip_code & zip_code_area

In [68]:
# mask
mask = (
        df_crime.x_coordinate.notna() & 
        df_crime.y_coordinate.notna() &
        df_crime.latitude.notna() &
        df_crime.longitude.notna() & 
            df_crime.zip_code.isna()
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Search: 108,455 :::::

--- Showing 5 of 108,455 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
7804970,2002,West Ridge,WEST RIDGE,98429094.8621,02,2,WEST RIDGE,98429094.8621,50,005,2,0512,<NA>,1179103,1834438,41.700986,-87.619801
222108,2024,West Ridge,WEST RIDGE,98429094.8621,29,2,WEST RIDGE,98429094.8621,24,010,4,1011,<NA>,1149242,1894205,41.865623,-87.727601
7685437,2002,West Ridge,WEST RIDGE,98429094.8621,02,2,WEST RIDGE,98429094.8621,50,005,2,0512,<NA>,1179103,1834438,41.700986,-87.619801
1049320,2020,West Ridge,WEST RIDGE,98429094.8621,11,2,WEST RIDGE,98429094.8621,45,016,5,1623,<NA>,1140596,1932113,41.969809,-87.758409
4357427,2009,West Ridge,WEST RIDGE,98429094.8621,35,2,WEST RIDGE,98429094.8621,03,002,1,0211,<NA>,1178135,1881804,41.830985,-87.621911


In [69]:
# Composite key
keys = ['year', 'x_coordinate', 'y_coordinate', 'latitude', 'longitude', 
        'district', 'sector', 'beat']
update_cols = ['community_code', 'primary_neighborhood', 'secondary_neighborhood', 'neighborhood_area',
               'ca_community_code', 'ca_community_name', 'ca_community_area', 'zip_code', 'zip_code_area']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 108,455
Rows updated:       2,894
Rows not updated:   105,561
Values imputed:     5,788


In [70]:
# Verify
df_crime.loc[idx, cols]

,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
7804970,2002,West Ridge,WEST RIDGE,98429094.8621,02,2,WEST RIDGE,98429094.8621,50,005,2,0512,<NA>,1179103,1834438,41.700986,-87.619801
222108,2024,West Ridge,WEST RIDGE,98429094.8621,29,2,WEST RIDGE,98429094.8621,24,010,4,1011,<NA>,1149242,1894205,41.865623,-87.727601
7685437,2002,West Ridge,WEST RIDGE,98429094.8621,02,2,WEST RIDGE,98429094.8621,50,005,2,0512,<NA>,1179103,1834438,41.700986,-87.619801
1049320,2020,West Ridge,WEST RIDGE,98429094.8621,11,2,WEST RIDGE,98429094.8621,45,016,5,1623,<NA>,1140596,1932113,41.969809,-87.758409
4357427,2009,West Ridge,WEST RIDGE,98429094.8621,35,2,WEST RIDGE,98429094.8621,03,002,1,0211,<NA>,1178135,1881804,41.830985,-87.621911


#### zip_code & zip_code_area & sector & district

In [71]:
# mask
mask = (
    df_crime.x_coordinate.notna() & 
    df_crime.y_coordinate.notna() &
    df_crime.zip_code.isna()
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Search: 105,561 :::::

--- Showing 5 of 105,561 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
4354691,2009,West Ridge,WEST RIDGE,98429094.8621,17,2,WEST RIDGE,98429094.8621,36,016,5,1631,<NA>,1125811,1925761,41.952638,-87.812916
668958,2022,West Ridge,WEST RIDGE,98429094.8621,02,2,WEST RIDGE,98429094.8621,49,024,3,2411,<NA>,1157830,1947717,42.012293,-87.69461
7697391,2002,West Ridge,WEST RIDGE,98429094.8621,02,2,WEST RIDGE,98429094.8621,50,005,2,0512,<NA>,1179103,1834438,41.700986,-87.619801
6279599,2005,West Ridge,WEST RIDGE,98429094.8621,67,2,WEST RIDGE,98429094.8621,17,007,1,0735,<NA>,1163874,1858853,41.768317,-87.674881
7797011,2002,West Ridge,WEST RIDGE,98429094.8621,02,2,WEST RIDGE,98429094.8621,50,024,3,2422,<NA>,1163435,1951532,42.022644,-87.673879


In [72]:
# Composite key
keys = ['x_coordinate', 'y_coordinate', 'latitude', 'longitude']
update_cols = ['sector', 'district', 'zip_code', 'zip_code_area']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 105,561
Rows updated:       83,852
Rows not updated:   21,709
Values imputed:     167,704


### Second Pass: Loosen Composite Key

In [73]:
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,543,089) ---
                        Count Percentage
zip_code_area           21709    0.2541%
zip_code                21709    0.2541%
secondary_description   19371    0.2267%
primary_description     19371    0.2267%
index_code              19371    0.2267%
location_description    15968    0.1869%
community_code           1312    0.0154%
district                  328    0.0038%
sector                    328    0.0038%
district_location         328    0.0038%
primary_neighborhood       71    0.0008%
secondary_neighborhood     71    0.0008%
neighborhood_area          71    0.0008%
ca_community_code          71    0.0008%
ca_community_name          71    0.0008%
ca_community_area          71    0.0008%
ward                       56    0.0007%


#### community_code & zip_code & zip_code_area

In [74]:
# mask
mask = (
    df_crime.x_coordinate.notna() & 
    df_crime.y_coordinate.notna() &
    df_crime.latitude.notna() &
    df_crime.longitude.notna()
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Search: 8,543,089 :::::

--- Showing 5 of 8,543,089 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
6005870,2005,South Chicago,SOUTH CHICAGO,93272185.0092,46,46,SOUTH CHICAGO,93272185.0092,10,004,2,0424,60617,1198076,1846205,41.732823,-87.549938
2904737,2013,Burnside,"CHATHAM,BURNSIDE",16995983.2737,47,47,BURNSIDE,16995983.2737,08,004,2,0413,60619,1184148,1844756,41.729183,-87.601007
5835722,2006,New City,BACK OF THE YARDS,134636963.254,61,61,NEW CITY,134636963.254,16,009,1,0932,60609,1166927,1868751,41.795414,-87.663407
3503885,2011,Riverdale,RIVERDALE,98389497.4143,54,54,RIVERDALE,98389497.4143,09,005,2,0533,60827,1185714,1818799,41.657917,-87.596084
4490856,2009,Humboldt Park,HUMBOLDT PARK,125010425.593,24,24,WEST TOWN,127562904.597,01,014,5,1421,60647,1157452,1910783,41.910951,-87.69701


In [75]:
# Composite key
keys = ['x_coordinate', 'y_coordinate', 'latitude', 'longitude']
update_cols = ['community_code', 'zip_code', 'zip_code_area']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 8,543,089
Rows updated:       1,140
Rows not updated:   8,541,949
Values imputed:     1,166


#### sector & district & zip_code & zip_code_area

In [76]:
# mask
mask = (
    df_crime.year.notna() &
    df_crime.x_coordinate.notna() & 
    df_crime.y_coordinate.notna() &
    df_crime.latitude.notna() &
    df_crime.longitude.notna()
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask, msg=False)

::::: Search: 8,543,089 :::::



In [77]:
# Composite key
keys = ['x_coordinate', 'secondary_neighborhood', 'y_coordinate', 'latitude', 'longitude']
update_cols = ['sector', 'district', 'zip_code', 'zip_code_area']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 8,543,089
Rows updated:       164
Rows not updated:   8,542,925
Values imputed:     328


#### sector & district & zip_code & zip_code_area & ward

In [78]:
# Composite key
keys = ['primary_neighborhood', 'district', 'beat']
update_cols = ['sector', 'district', 'zip_code', 'zip_code_area', 'ward']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 8,543,089
Rows updated:       21,895
Rows not updated:   8,521,194
Values imputed:     43,570


#### ca_community_code & ca_community_name & ca_community_area & zip_code & zip_code_area

In [79]:
# Composite key
keys = ['district_location', 'community_code', 'district', 'primary_neighborhood']
update_cols = ['ca_community_code', 'ca_community_name', 'ca_community_area', 'zip_code', 'zip_code_area']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 8,543,089
Rows updated:       78
Rows not updated:   8,543,011
Values imputed:     227


#### primary_neighborhood & secondary_neighborhood

In [80]:
# Composite key
keys = ['beat', 'district', 'sector']
update_cols = ['district_location','community_code', 'primary_neighborhood', 'neighborhood_area', 
               'secondary_neighborhood', 'ca_community_code', 'ca_community_name', 'ca_community_area',
               'zip_code', 'zip_code_area']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 8,543,089
Rows updated:       433
Rows not updated:   8,542,656
Values imputed:     575


### Last Clean-Up: Impute

In [81]:
# nans
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,543,089) ---
                       Count Percentage
primary_description    19371    0.2267%
secondary_description  19371    0.2267%
index_code             19371    0.2267%
location_description   15968    0.1869%
zip_code                   1    0.0000%
zip_code_area              1    0.0000%


In [82]:
# Composite key
keys = ['beat']
update_cols = ['zip_code', 'zip_code_area']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 8,543,089
Rows updated:       1
Rows not updated:   8,543,088
Values imputed:     2


### DataFrame Maint

In [83]:
# collapse fragmented blocks
df_crime = df_crime.copy()
# display
df_crime.head()

,case_number,date,block,iucr,primary_description,secondary_description,index_code,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_code,year,updated_on,fbi_code,zip_code,zip_code_area,primary_neighborhood,secondary_neighborhood,neighborhood_area,sector,ca_community_code,ca_community_name,ca_community_area,latitude,longitude,x_coordinate,y_coordinate,era,month,day_of_week,quarter,year_quarter,time_of_day,fbi_code_desc,fbi_index_code,district_location
8265318,01G050460,2001-01-24 20:45:00,072XX S RIDGELAND AV,1811,NARCOTICS,POSSESS - CANNABIS 30 GRAMS OR LESS,N,NARCOTICS,POSS: CANNABIS 30GMS OR LESS,SIDEWALK,True,False,0324,003,50,02,2001,2015-08-17 15:03:40,18,60649,80526075.8505,South Shore,"SOUTH SHORE, GRAND CROSSING",81812716.3904,1,43,SOUTH SHORE,81812716.3958,41.764219,-87.582549,1189075,1857566,pre_covid,January,Wednesday,Q1,2001-Q1,Night,Drug Abuse Violations,False,Grand Crossing
7080787,03J493690,2003-07-12 17:00:00,075XX S DOBSON AVE,0890,THEFT,FROM BUILDING,I,THEFT,FROM BUILDING,APARTMENT,False,False,0624,006,08,69,2003,2015-08-17 15:03:40,06,60619,167872012.337,West Ridge,WEST RIDGE,98429094.8621,2,2,WEST RIDGE,98429094.8621,41.755291,-87.59816,1184844,1854276,pre_covid,July,Saturday,Q3,2003-Q3,Evening,Larceny – Theft,True,Gresham
6396203,04X245238,2004-12-13 21:15:00,006XX N RIDGEWAY AVE,2024,NARCOTICS,POSSESS - HEROIN (WHITE),N,NARCOTICS,POSS: HEROIN(WHITE),SIDEWALK,True,False,1122,011,27,23,2004,2018-02-28 15:56:25,18,60624,99418122.6738,Humboldt Park,HUMBOLDT PARK,125010425.593,4,23,HUMBOLDT PARK,100480876.502,41.892451,-87.719888,1151273,1903996,pre_covid,December,Monday,Q4,2004-Q4,Night,Drug Abuse Violations,False,Harrison
5818733,07C115980,2006-03-31 09:15:00,026XX N NARRAGANSETT AVE,0610,BURGLARY,FORCIBLE ENTRY,I,BURGLARY,FORCIBLE ENTRY,APARTMENT,False,False,2512,025,29,19,2006,2018-02-28 15:56:25,05,60707,48519709.6539,Belmont Cragin,"BELMONT CRAGIN,HERMOSA",109099407.211,5,19,BELMONT CRAGIN,109099414.689,41.928096,-87.78561,1133296,1916864,pre_covid,March,Friday,Q1,2006-Q1,Morning,Burglary,True,Grand Central
5312174,07HN36467,2007-05-25 14:51:00,022XX N LA CROSSE AVE,1812,NARCOTICS,POSSESS - CANNABIS MORE THAN 30 GRAMS,N,NARCOTICS,POSS: CANNABIS MORE THAN 30GMS,RESIDENCE,True,False,2522,025,31,19,2007,2018-02-28 15:56:25,18,60639,127476051.26,Belmont Cragin,"BELMONT CRAGIN,HERMOSA",109099407.211,5,19,BELMONT CRAGIN,109099414.689,41.921066,-87.747452,1143697,1914371,pre_covid,May,Friday,Q2,2007-Q2,Afternoon,Drug Abuse Violations,False,Grand Central


#### Examine Each NaNs

In [84]:
# Final State
print(f"Shape: {df_crime.shape}")
print(f"\nDtypes:\n{df_crime.dtypes}")
utils.any_nans(df_crime)

Shape: (8543089, 41)

Dtypes:
case_number                                                 string[pyarrow]
date                                                  timestamp[s][pyarrow]
block                                                       string[pyarrow]
iucr                                                        string[pyarrow]
primary_description                                         string[pyarrow]
secondary_description                                       string[pyarrow]
index_code                                                  string[pyarrow]
primary_type                                                string[pyarrow]
description                                                 string[pyarrow]
location_description                                        string[pyarrow]
arrest                                                                 bool
domestic                                                               bool
beat                                                      

* Decision: retain rows - nulls preserved for auditability and future validation.
* Downstream analysis does not depend on community_code or ward features.

## Total Time

In [85]:
# total elapsed time
elapsed = time.time() - start
print(f"{elapsed:.2f}s to process {df_crime.shape[0]:,} rows")

90.93s to process 8,543,089 rows


## Save using PyArrow

In [86]:
# Save as Arrow
feather.write_feather(df_crime, "../Data/crime_data.feather")